In [0]:

-- ============================================================
-- SECTION 1: ROW COUNT VALIDATION
-- ============================================================

-- Check 1: Bronze Layer Row Counts
SELECT 'customers_raw' AS table_name, COUNT(*) AS row_count
FROM `retail-dwh-project`.bronze.customers_raw

UNION ALL

SELECT 'products_raw', COUNT(*)
FROM `retail-dwh-project`.bronze.products_raw

UNION ALL

SELECT 'stores_raw', COUNT(*)
FROM `retail-dwh-project`.bronze.stores_raw

UNION ALL

SELECT 'sales_raw', COUNT(*)
FROM `retail-dwh-project`.bronze.sales_raw;


-- Check 2: Silver Layer Row Counts
SELECT 'DimCustomer' AS table_name, COUNT(*) AS row_count
FROM `retail-dwh-project`.silver.DimCustomer

UNION ALL

SELECT 'DimProduct', COUNT(*)
FROM `retail-dwh-project`.silver.DimProduct

UNION ALL

SELECT 'DimStore', COUNT(*)
FROM `retail-dwh-project`.silver.DimStore

UNION ALL

SELECT 'FactSales', COUNT(*)
FROM `retail-dwh-project`.silver.FactSales;


-- ============================================================
-- SECTION 2: DIMCUSTOMER VALIDATION
-- ============================================================

-- Check 1: Uppercase emails → Expected: 0
SELECT COUNT(*) AS uppercase_email_count
FROM `retail-dwh-project`.silver.DimCustomer
WHERE Email != LOWER(Email);


-- Check 2: CustomerName not in Proper Case → Expected: 0
SELECT COUNT(*) AS improper_name_count
FROM `retail-dwh-project`.silver.DimCustomer
WHERE CustomerName != INITCAP(CustomerName);


-- Check 3: City with leading/trailing spaces → Expected: 0
SELECT COUNT(*) AS city_spaces_count
FROM `retail-dwh-project`.silver.DimCustomer
WHERE City != TRIM(City);


-- Check 4: Duplicate active CustomerIDs → Expected: 0
SELECT COUNT(*) AS duplicate_active_customers
FROM (
    SELECT CustomerID, COUNT(*) AS cnt
    FROM `retail-dwh-project`.silver.DimCustomer
    WHERE IsActive = 1
    GROUP BY CustomerID
    HAVING cnt > 1
);


-- Check 5: Duplicate CustomerSK → Expected: 0
SELECT COUNT(*) AS duplicate_sk_count
FROM (
    SELECT CustomerSK, COUNT(*) AS cnt
    FROM `retail-dwh-project`.silver.DimCustomer
    GROUP BY CustomerSK
    HAVING cnt > 1
);


-- ============================================================
-- SECTION 3: SCD TYPE 2 VALIDATION
-- ============================================================

-- Check 1: Active vs Inactive Customer Summary
SELECT IsActive, COUNT(*) AS count
FROM `retail-dwh-project`.silver.DimCustomer
GROUP BY IsActive;


-- Check 2: Inactive records with wrong EndDate → Expected: 0
SELECT COUNT(*) AS wrong_enddate_count
FROM `retail-dwh-project`.silver.DimCustomer
WHERE IsActive = 0
  AND EndDate = DATE('9999-12-31');


-- Check 3: Active records without default EndDate → Expected: 0
SELECT COUNT(*) AS wrong_active_enddate
FROM `retail-dwh-project`.silver.DimCustomer
WHERE IsActive = 1
  AND EndDate != DATE('9999-12-31');


-- Check 4: NULL StartDate → Expected: 0
SELECT COUNT(*) AS null_startdate_count
FROM `retail-dwh-project`.silver.DimCustomer
WHERE StartDate IS NULL;


-- Check 5: Full SCD2 History
SELECT CustomerID,
       CustomerName,
       City,
       Address,
       StartDate,
       EndDate,
       IsActive
FROM `retail-dwh-project`.silver.DimCustomer
WHERE CustomerID IN (
    SELECT CustomerID
    FROM `retail-dwh-project`.silver.DimCustomer
    GROUP BY CustomerID
    HAVING COUNT(*) > 1
)
ORDER BY CustomerID, StartDate;


-- ============================================================
-- SECTION 4: DIMPRODUCT VALIDATION
-- ============================================================

-- Check 1: UnitPrice = 0 → Expected: 0
SELECT COUNT(*) AS zero_price_count
FROM `retail-dwh-project`.silver.DimProduct
WHERE UnitPrice = 0;


-- Check 2: Zero-price products in Bronze
SELECT COUNT(*) AS rejected_from_bronze
FROM `retail-dwh-project`.bronze.products_raw
WHERE UnitPrice = 0;


-- Check 3: ProductName with extra spaces → Expected: 0
SELECT COUNT(*) AS product_name_spaces
FROM `retail-dwh-project`.silver.DimProduct
WHERE ProductName != TRIM(ProductName);


-- Check 4: Category with extra spaces → Expected: 0
SELECT COUNT(*) AS category_spaces
FROM `retail-dwh-project`.silver.DimProduct
WHERE Category != TRIM(Category);


-- Check 5: Duplicate ProductSK → Expected: 0
SELECT COUNT(*) AS duplicate_product_sk
FROM (
    SELECT ProductSK, COUNT(*) AS cnt
    FROM `retail-dwh-project`.silver.DimProduct
    GROUP BY ProductSK
    HAVING cnt > 1
);


-- ============================================================
-- SECTION 5: DIMSTORE VALIDATION
-- ============================================================

-- Check 1: NULL Region → Expected: 0
SELECT COUNT(*) AS null_region_count
FROM `retail-dwh-project`.silver.DimStore
WHERE Region IS NULL;


-- Check 2: NULL Regions in Bronze Source
SELECT COUNT(*) AS null_region_in_bronze
FROM `retail-dwh-project`.bronze.stores_raw
WHERE Region IS NULL
   OR TRIM(Region) = '';


-- Check 3: StoreName not in Proper Case → Expected: 0
SELECT COUNT(*) AS improper_storename_count
FROM `retail-dwh-project`.silver.DimStore
WHERE StoreName != INITCAP(StoreName);


-- Check 4: StoreName with extra spaces → Expected: 0
SELECT COUNT(*) AS storename_spaces
FROM `retail-dwh-project`.silver.DimStore
WHERE StoreName != TRIM(StoreName);


-- Check 5: Duplicate StoreSK → Expected: 0
SELECT COUNT(*) AS duplicate_store_sk
FROM (
    SELECT StoreSK, COUNT(*) AS cnt
    FROM `retail-dwh-project`.silver.DimStore
    GROUP BY StoreSK
    HAVING cnt > 1
);


-- ============================================================
-- SECTION 6: FACTSALES VALIDATION
-- ============================================================

-- Check 1: Duplicate TransactionIDs → Expected: 0
SELECT COUNT(*) AS duplicate_txn_count
FROM (
    SELECT TransactionID, COUNT(*) AS cnt
    FROM `retail-dwh-project`.silver.FactSales
    GROUP BY TransactionID
    HAVING cnt > 1
);


-- Check 2: Quantity = 0 → Expected: 0
SELECT COUNT(*) AS zero_qty_count
FROM `retail-dwh-project`.silver.FactSales
WHERE Quantity = 0;


-- Check 3: NULL SKs in FactSales → Expected: 0
SELECT COUNT(*) AS null_sk_count
FROM `retail-dwh-project`.silver.FactSales
WHERE CustomerSK IS NULL
   OR ProductSK  IS NULL
   OR StoreSK    IS NULL;


-- Check 4: Amount Validation → Expected: PASS
SELECT
    f.TransactionID,
    f.Quantity,
    p.UnitPrice,
    f.Amount,
    ROUND(f.Quantity * p.UnitPrice, 2) AS Expected_Amount,
    CASE
        WHEN f.Amount = ROUND(f.Quantity * p.UnitPrice, 2)
        THEN 'PASS'
        ELSE 'FAIL'
    END AS Amount_Check
FROM `retail-dwh-project`.silver.FactSales f
JOIN `retail-dwh-project`.silver.DimProduct p
    ON f.ProductSK = p.ProductSK
LIMIT 10;


-- Check 5: Incorrect TxnDate Format → Expected: 0
SELECT COUNT(*) AS wrong_date_format
FROM `retail-dwh-project`.silver.FactSales
WHERE TxnDate NOT RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}$';
